# 4.9b MCP 原理平替版(零依赖 · 纯 Python)

> ⚠️ **这个 notebook 不需要装任何库,不需要 asyncio,不需要 GPU** —— 直接运行全部 cell 即可。

## 它和 4.9 实战版有什么区别?

| | 4.9 实战版 | 4.9b 平替版(本文件) |
|---|---|---|
| 依赖 | 装 `mcp` + `openai` 库 | **零依赖**,纯标准库 |
| 通信 | 真起子进程(stdin/stdout) | **用 Python 类模拟** |
| 模型 | 真调 Qwen API | **用假模型**演示 |
| 目的 | 能跑的工程代码 | **看懂 MCP 在干什么** |

**这个版本的目标**:让你看清 MCP 的**协议流程** —— 客户端怎么发现工具、怎么调用、结果怎么回来。它不真连服务器,但把每一步都"演"给你看。

## 0. 用生活类比理解 MCP

想象你去一家**餐厅点菜**:

```
你(LLM 大模型)        服务员(MCP Client)            厨房(MCP Server)
   │                       │                              │
   │   "我要点菜"          │                              │
   │ ────────────────────> │   "你们有什么菜?"(list_tools) │
   │                       │ ────────────────────────────> │
   │                       │   "我们有: 回锅肉/番茄炒蛋"   │
   │                       │ <──────────────────────────── │
   │   "那来份回锅肉"      │                              │
   │ ────────────────────> │   "做一份回锅肉"(call_tool)   │
   │                       │ ────────────────────────────> │
   │                       │   "回锅肉做好了"              │
   │                       │ <──────────────────────────── │
   │   "哦,回锅肉是辣的!"  │                              │
   │ <──────────────────── │                              │
```

**关键点**:你(大模型)**不需要知道厨房怎么炒菜**(工具怎么实现),你只需要:
1. 知道**菜单上有啥**(工具列表 = `list_tools`)
2. 报菜名点菜(调用工具 = `call_tool`)

MCP 就是这套**点菜协议**。客户端负责传话,服务器负责做菜。

## 1. 模拟一个 MCP 服务器(Server 端)

真实 MCP Server 会注册工具。这里用**一个 Python 类**模拟 —— 它"知道"有哪些工具、怎么执行,就像厨房知道怎么炒菜。

In [1]:
# 模拟 MCP 服务器:它内部有真正的工具实现
class MockServer:
    """模拟 MCP Server —— 相当于厨房,里面才是真正的工具"""
    def __init__(self):
        # 工具注册表:名字 -> 函数
        self.tools = {
            "get_current_weather": self._get_weather,
            "add": self._add,
        }
        self._weather_db = {
            "北京": "晴 32°C",
            "杭州": "小雨 27°C",
            "上海": "多云 30°C",
        }

    # ===== 工具实现(真正干活的地方)=====
    def _get_weather(self, location: str) -> str:
        key = location.rstrip("市")
        return f"{location}: {self._weather_db.get(key, '未知')}"

    def _add(self, a: int, b: int) -> str:
        return str(a + b)

    # ===== MCP 协议的两个核心方法 =====
    def list_tools(self) -> dict:
        """告诉客户端:我能干什么(相当于给菜单)"""
        return {name: {"type": "function", "description": f"工具 {name}"} for name in self.tools}

    def call_tool(self, name: str, arguments: dict) -> str:
        """执行工具并返回结果(相当于做菜)"""
        if name not in self.tools:
            return f"错误: 没有这个工具 {name}"
        return self.tools[name](**arguments)


# 实例化服务器
server = MockServer()
print("✅ 模拟 MCP 服务器已就绪")

✅ 模拟 MCP 服务器已就绪


## 2. 模拟一个 MCP 客户端(Client 端)

客户端**不实现工具**,它只做三件事:
1. 连接服务器
2. 问服务器"你有啥工具"(`list_tools`)
3. 叫服务器"帮我执行这个工具"(`call_tool`)

这就是笔记里说的:**客户端动态发现工具,不用预先知道有哪些函数**。

In [2]:
# 模拟 MCP 客户端:它不干活,只负责传话
class MockClient:
    """模拟 MCP Client —— 相当于服务员,自己不炒菜,只传话"""
    def __init__(self, server):
        self.server = server   # 连上了服务器
        self.discovered_tools = {}  # 动态发现的工具缓存

    def connect(self):
        """1) 握手 + 动态发现工具(客户端此刻才知道服务器有啥)"""
        self.discovered_tools = self.server.list_tools()
        print(f"🔌 已连接服务器,动态发现 {len(self.discovered_tools)} 个工具:")
        for name in self.discovered_tools:
            print(f"   - {name}")
        return self.discovered_tools

    def call(self, name: str, arguments: dict) -> str:
        """2) 调用服务器上的工具(客户端自己不会实现!)"""
        if name not in self.discovered_tools:
            return f"错误: 没发现过这个工具 {name}"
        result = self.server.call_tool(name, arguments)
        print(f"🤖 客户端说: 请执行 {name}{arguments}")
        print(f"🛠️  服务器返回: {result}")
        return result


# 创建客户端并连接
client = MockClient(server)
client.connect()

🔌 已连接服务器,动态发现 2 个工具:
   - get_current_weather
   - add


{'get_current_weather': {'type': 'function',
  'description': '工具 get_current_weather'},
 'add': {'type': 'function', 'description': '工具 add'}}

## 3. 模拟"大模型":它会自己决定调不调工具

真实场景里,是**大模型**决定要不要调工具。这里用一个**假模型**模拟这个过程 —— 它按规则"假装思考",然后要么直接回答,要么输出要调用的工具。

In [3]:
# 模拟大模型:决定"要不要调工具"
class MockLLM:
    """模拟大模型 —— 它会判断用户问题是否需要外部工具"""
    def __init__(self, client):
        self.client = client
        # 假模型的"知识库"(它自己知道的)
        self.known = {
            "什么是MCP": "MCP是模型上下文协议,统一了Agent工具调用规范。",
            "1+1等于几": "等于2。",
        }

    def process(self, question: str) -> str:
        """核心: 判断 → 调工具(若需要) → 回答"""
        # 1) 模型判断:这个问题我知道吗?
        if question in self.known:
            return f"[模型直接回答] {self.known[question]}"

        # 2) 不知道 → 决定调用工具
        print(f"🧠 模型: '{question}' 超出我的知识范围,需要调工具!")
        if "天气" in question:
            # 从问题里提取城市(真实场景由模型解析,这里简化)
            for city in ["北京", "上海", "杭州"]:
                if city in question:
                    result = self.client.call("get_current_weather", {"location": city})
                    return f"[模型基于工具结果] {result}"

        if "加" in question or "+" in question:
            result = self.client.call("add", {"a": 3, "b": 5})
            return f"[模型基于工具结果] 结果是 {result}"

        return "[模型] 我不知道,也没有合适的工具。"


llm = MockLLM(client)

# 测试1: 模型自己知道的,不调工具
print("用户问: 什么是MCP?")
print(llm.process("什么是MCP"))
print()

用户问: 什么是MCP?
[模型直接回答] MCP是模型上下文协议,统一了Agent工具调用规范。



In [4]:
# 测试2: 超出知识范围,调天气工具
print("用户问: 北京今天天气怎么样?")
print(llm.process("北京今天天气怎么样"))
print()

用户问: 北京今天天气怎么样?
🧠 模型: '北京今天天气怎么样' 超出我的知识范围,需要调工具!
🤖 客户端说: 请执行 get_current_weather{'location': '北京'}
🛠️  服务器返回: 北京: 晴 32°C
[模型基于工具结果] 北京: 晴 32°C



In [5]:
# 测试3: 需要计算,调加法工具
print("用户问: 帮我算一下 3 + 5 等于多少?")
print(llm.process("帮我算一下 3 + 5 等于多少?"))

用户问: 帮我算一下 3 + 5 等于多少?
🧠 模型: '帮我算一下 3 + 5 等于多少?' 超出我的知识范围,需要调工具!
🤖 客户端说: 请执行 add{'a': 3, 'b': 5}
🛠️  服务器返回: 8
[模型基于工具结果] 结果是 8


## 4. 为什么需要 MCP?(对比裸写)

把上面简化,真实 MCP 的价值体现在**两个对比**:

### 对比1: 客户端要不要知道工具实现?
```python
# ❌ 没有 MCP:客户端要自己实现天气函数
def my_client_question(question):
    if "天气" in question:
        # 客户端自己写一遍天气查询逻辑... 每个客户端都要写一遍
        return query_weather_impl()   # 硬编码!

# ✅ 有 MCP:客户端只调 call_tool
result = client.call("get_current_weather", {"location": "北京"})
# 实现全在服务器,客户端不重复写
```

### 对比2: 换个服务器要不要改客户端?
```python
# ✅ 客户端代码一行不用改,只换连接的服务器
client = MockClient(server_A)   # 连天气服务器
client = MockClient(server_B)   # 连数据库服务器,代码一样!
# 因为工具都是通过 list_tools 动态发现的
```

**这就是笔记说的**:MCP 把工具注册标准化成协议,同一个客户端可以连任何符合规范的服务器,避免外部函数重复编写。

## 📊 总结

### 一张图记住 MCP 流程
```
用户提问 → 大模型判断 → (超出知识范围?)
                            │ 是
                            ▼
                 客户端 list_tools 发现工具
                            │
                            ▼
                 客户端 call_tool 调用工具
                            │
                            ▼
                 服务器执行,返回结果
                            │
                            ▼
                 大模型基于结果生成最终答案
```

### 三个核心角色
| 角色 | 比喻 | 职责 |
|------|------|------|
| **LLM(大模型)** | 顾客 | 判断要不要点菜,消费结果 |
| **MCP Client** | 服务员 | 传话,list_tools + call_tool |
| **MCP Server** | 厨房 | 真正实现工具,返回结果 |

### 一句话
> MCP = **统一的点菜协议**。大模型只需要知道"有哪些工具、怎么调用",不需要关心工具内部实现 —— 工具由服务器提供,客户端动态发现,换服务器不用改代码。

### 下一步想学更深的?
- 本文件是**平替版**(看懂原理)
- 想跑真实代码 → 看 [4.9 MCP 客户端实战](4.9 MCP 客户端实战.ipynb)(真装库、真起子进程)
- 想部署本地模型 → 看 [4.10 本地模型部署实战](4.10 本地模型部署实战.ipynb)